In [6]:
%pip install -q google-play-scraper pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [7]:
import warnings
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from google_play_scraper import Sort, reviews

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# Scrapping review aplikasi Duolingo
APP_ID = "com.levelinfinite.sgameGlobal"
APP_NAME = "Honor of King"
LANG = "id"
COUNTRY = "id"
TARGET_COUNT = 10000
MAX_BATCH = 2000

OUTPUT_CSV = "reviews_hok.csv"

print(f"Target total sampel: ~{TARGET_COUNT}".replace(",", "."))

Target total sampel: ~10000


In [8]:
def scrape_reviews_for_app(app_id: str, app_name: str, target_count: int = 2000):
    """Scrape review sampai target terpenuhi atau data habis."""
    all_rows = []
    token = None

    while len(all_rows) < target_count:
        batch_count = min(MAX_BATCH, target_count - len(all_rows))
        result, token = reviews(
            app_id,
            lang=LANG,
            country=COUNTRY,
            sort=Sort.MOST_RELEVANT,
            count=batch_count,
            continuation_token=token,
        )

        if not result:
            break

        for r in result:
            all_rows.append(
                {
                    "review_id": r.get("reviewId"),
                    "user_name": r.get("userName"),
                    "score": r.get("score"),
                    "content": r.get("content"),
                    "thumbs_up": r.get("thumbsUpCount"),
                    "app_version": r.get("appVersion"),
                }
            )

        if token is None:
            break

    return all_rows

In [9]:
all_reviews = scrape_reviews_for_app(APP_ID, APP_NAME, target_count=TARGET_COUNT)

raw_df = pd.DataFrame(all_reviews)
print(f"\nTotal content terkumpul (raw): {len(raw_df):,}".replace(",", "."))
raw_df.head()


Total content terkumpul (raw): 10.000


,review_id,user_name,score,content,thumbs_up,app_version
0,2c793cc6-a533-4756-899c-929fe0905f1f,Muhammad Ramadani,5,"game nya seru, developer nya baik, heronya balance daripada moba lain. hal yang aku permasalahkan; sering kali layar...",11,11.3.1.8
1,03d587fa-daa9-4a40-b7c0-31ad6207ec45,Andriyanto,2,"tolong perbaiki peforma peningkatan sinyal nya, kadang yang diutamakan itu pengalaman bermain, tolong perbaiki serve...",2,11.3.1.9
2,f9f2926a-cfd2-4c77-b668-48813c845d28,Andika,2,"gamenya udah bagus cuma ada kekurangan di fitur download, yaitu skin dalam match (yang dimiliki) sama yang (belum di...",1,11.3.1.9
3,6febf65b-1bcd-4893-879a-03b7679ea16c,icooo,1,"game rusak game rusak, lain kali kalo update jangan cuma mikirin skin,collab,dll. pentingin juga optimalisasi jaring...",0,11.3.1.9
4,7a0f84f8-5c43-497a-b6c2-35b045d29061,Nur Juminati,5,"Suka banget sama game nya, soalnya hero nya juga kebanyakan cakep-cakep:b. Cuma ya.. Kekurangan nya itu cuma satu, j...",1,11.3.1.9


In [11]:
df = raw_df.copy()

print("Distribusi skor:")
display(df["score"].value_counts().sort_index())

# Simpan ke CSV
ordered_cols = [
    "review_id", "user_name", "score", "content", "thumbs_up", "app_version"
]
existing_cols = [c for c in ordered_cols if c in df.columns]
df[existing_cols].to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"\nFile CSV tersimpan di: {OUTPUT_CSV}")
print(f"Jumlah sampel akhir: {len(df):,}".replace(",", "."))

Distribusi skor:


score
1    3295
2     979
3    1121
4    1250
5    3355
Name: count, dtype: int64


File CSV tersimpan di: reviews_hok.csv
Jumlah sampel akhir: 10.000


In [13]:
df.sample(5, random_state=42)[["user_name", "score", "content"]]

,user_name,score,content
6252,Joe Vinsend,5,"Gamenya bagus loh, aku suka saat ada skin gratis, walaupun aku suka top up, tapi saat ghaca di kasih yang bagus, kec..."
4684,andre rmxmzyuu,1,upgrade dong animasi jalan Ama tempur nya Masi kaku ini gimana game kaku kalah Ama ml minimal upgrade dulu semua ani...
1731,Rizma Usabillillah,3,"entah kenapa saya merasa kurang enak mendengar suara di game ini, walaupun Hero, Mob dan sebagainya masih terlihat k..."
4742,Rizqi Ramdhan,1,"Tolong di perhatikan kenyamanan player yang bermain, server up and down meski sudah pake wifi terkencsng sekali pun,..."
4521,Doge,5,"Ini game sebenernya sih udah bagus banget, memanjakan mata dengan grafik dan efek dalam match-nya, menyenangkan hati..."
